# Bloque de validación · ¿Es un criterio? ¿Dice lo mismo que la cita?

**Sesión Norma+ · Diplomado de Ciencia de Datos e IA para Ciencias Sociales**

El Lab 2 terminó con una advertencia: una cita literal garantiza que el texto *existe*, no que sea un criterio. En este bloque agregamos dos controles que Norma+ aplica después de extraer:

| Control | Pregunta | Qué descarta |
|---|---|---|
| **Validador** | ¿Este fragmento es un criterio jurídico? | Hechos, transcripciones, argumentos de parte, valoración de prueba, trámites, aplicación al caso |
| **Gate de fidelidad** | ¿La abstracción dice lo mismo que la cita? | Generalizaciones que el texto no sostiene |

Antes de confiar en un control, **lo medimos** contra datos etiquetados.

Modelo: `claude-haiku-4-5`. Costo esperado del bloque: menos de USD 0.05.

## 0 · Preparación

In [ ]:
!pip install -q anthropic pydantic pandas

In [ ]:
# Descarga del dataset desde el repositorio público del curso (git viene instalado en Colab).
import os, shutil, zipfile
REPO_URL = "https://github.com/benjov/laboratorio-norma-notebooks"

if not os.path.exists("norma_mini_dataset"):
    !git clone -q --depth 1 "$REPO_URL" repo_curso
    if os.path.exists("repo_curso/norma_mini_dataset"):
        shutil.copytree("repo_curso/norma_mini_dataset", "norma_mini_dataset")
    else:
        print("No se pudo descargar de GitHub. Sube norma_mini_dataset.zip a mano:")
        from google.colab import files
        files.upload()
        zipfile.ZipFile("norma_mini_dataset.zip").extractall(".")
print(sorted(os.listdir("norma_mini_dataset")))

In [ ]:
from google.colab import userdata
import anthropic
client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

In [ ]:
# Guardián de presupuesto: envuelve al cliente, suma el costo de cada llamada y se detiene al llegar al límite.
# Precios en USD por millón de tokens (entrada, salida). Un modelo no listado usa un precio conservador.
PRECIOS = {"claude-haiku-4-5": (1.0, 5.0)}
PRECIO_CONSERVADOR = (5.0, 25.0)

class Presupuesto:
    def __init__(self, client, limite_usd):
        self.client, self.limite, self.gastado, self.llamadas = client, limite_usd, 0.0, 0
        self.registro = []  # una fila por llamada: modelo, tokens y costo

    def crear(self, **kwargs):
        if self.gastado >= self.limite:
            raise RuntimeError(f"Límite de USD {self.limite:.2f} alcanzado en este notebook.")
        r = self.client.messages.create(**kwargs)
        p_in, p_out = PRECIOS.get(kwargs["model"], PRECIO_CONSERVADOR)
        costo = (r.usage.input_tokens * p_in + r.usage.output_tokens * p_out) / 1e6
        self.gastado += costo
        self.llamadas += 1
        self.registro.append({"modelo": kwargs["model"], "entrada": r.usage.input_tokens,
                              "salida": r.usage.output_tokens, "costo": costo})
        return r

    def __repr__(self):
        return f"{self.llamadas} llamadas · USD {self.gastado:.4f} de {self.limite:.2f}"

In [ ]:
import re, json, hashlib
from pathlib import Path
from typing import Literal
import pandas as pd
from pydantic import BaseModel, Field, ValidationError

DATA = Path("norma_mini_dataset")
SALIDAS = DATA / "salidas"
SALIDAS.mkdir(exist_ok=True)
MODELO = "claude-haiku-4-5"
api = Presupuesto(client, limite_usd=0.50)

gold = pd.read_csv(DATA / "gold_criterios.csv")
negativos = pd.read_csv(DATA / "negativos_validacion.csv")
pares = pd.read_csv(DATA / "pares_fidelidad.csv")
chunks = [json.loads(l) for l in open(SALIDAS / "chunks_lab1.jsonl", encoding="utf-8")]

CACHE_RUTA = SALIDAS / "cache_lab2b.json"
cache = json.load(open(CACHE_RUTA, encoding="utf-8")) if CACHE_RUTA.exists() else {}

def llamar(sistema, usuario, herramienta, max_tokens=400):
    """Llamada con esquema forzado y caché por hash del contenido."""
    clave = hashlib.sha256((MODELO + sistema + usuario).encode()).hexdigest()
    if clave not in cache:
        r = api.crear(model=MODELO, max_tokens=max_tokens, system=sistema, tools=[herramienta],
                      tool_choice={"type": "tool", "name": herramienta["name"]},
                      messages=[{"role": "user", "content": usuario}])
        cache[clave] = next(b.input for b in r.content if b.type == "tool_use")
        json.dump(cache, open(CACHE_RUTA, "w", encoding="utf-8"), ensure_ascii=False)
    return cache[clave]

def normalizar(s):
    return re.sub(r"\s+", " ", s).strip()

def contexto(documento, fragmento):
    """El apartado completo donde aparece el fragmento: el validador necesita ver el entorno."""
    f = normalizar(fragmento)
    return next((c for c in chunks if c["documento"] == documento and f in c["texto"]), None)

print(len(gold), "positivos ·", len(negativos), "negativos ·", len(pares), "pares de fidelidad ·", api)

## 1 · El validador

Le mostramos al modelo un fragmento **y el apartado donde aparece**, y le pedimos que lo clasifique en una de siete categorías. Pedimos además una confianza y una razón breve: la razón sirve para auditar, la confianza para decidir.

In [ ]:
TIPOS = ("criterio", "hecho", "transcripcion", "argumento_parte", "valoracion_prueba", "aplicacion_al_caso", "tramite")

class Veredicto(BaseModel):
    tipo: Literal[TIPOS] = Field(description="Categoría del fragmento.")
    confianza: float = Field(description="Entre 0 y 1: qué tan seguro estás de la categoría.")
    razon: str = Field(description="Una oración que justifique la categoría.")

SISTEMA_VAL = """Eres un validador jurídico. Clasifica el FRAGMENTO, leído dentro de su APARTADO, en una categoría:
- criterio: proposición general sobre cómo se interpreta o aplica una norma, invocable en otro caso;
- hecho: dato del caso (fechas, montos, porcentajes, conductas concretas);
- transcripcion: texto copiado de un contrato, ley u otro documento;
- argumento_parte: lo que sostiene una de las partes;
- valoracion_prueba: lo que la autoridad concluye de pruebas concretas;
- aplicacion_al_caso: un criterio aplicado a los hechos de este caso;
- tramite: actuaciones del procedimiento.
Si dudas entre criterio y otra categoría, elige la otra."""

def esquema(modelo):
    s = modelo.model_json_schema(); s.pop("$defs", None); return s

HERR_VAL = {"name": "clasificar_fragmento", "description": "Registra la categoría del fragmento.",
            "input_schema": esquema(Veredicto)}

def validar(documento, fragmento):
    ctx = contexto(documento, fragmento)
    usuario = f"APARTADO ({ctx['titulo'] if ctx else 's/d'}):\n{ctx['texto'] if ctx else ''}\n\nFRAGMENTO:\n{fragmento}"
    try:
        return Veredicto.model_validate(llamar(SISTEMA_VAL, usuario, HERR_VAL))
    except ValidationError:
        return None

### ✏️ Ejercicio 1 · La regla de decisión

El validador da un veredicto; nosotros decidimos qué hacer con él. Completa `aceptar` para que **falle cerrado**: solo acepta un fragmento si el veredicto existe, su tipo es `criterio` y la confianza es al menos `umbral`. Cualquier otra cosa se descarta.

In [ ]:
def aceptar(veredicto, umbral=0.7):
    return veredicto is not None and veredicto.tipo == "criterio" and veredicto.confianza >= umbral

In [ ]:
class _V:  # veredictos de prueba
    def __init__(self, tipo, confianza): self.tipo, self.confianza = tipo, confianza
assert aceptar(_V("criterio", 0.9)) and not aceptar(_V("criterio", 0.5))
assert not aceptar(_V("hecho", 0.99)) and not aceptar(None)
print("✅ Regla de decisión correcta")

### Medir el validador

Armamos un conjunto etiquetado: las 22 citas del gold (positivos) y los 8 fragmentos negativos. Si el validador no distingue bien *aquí*, donde sabemos la respuesta, no podemos confiar en él sobre extracciones nuevas.

In [ ]:
etiquetado = pd.concat([
    gold.assign(texto=gold.cita_literal, real="criterio")[["documento", "texto", "real"]],
    negativos.assign(texto=negativos.texto_literal, real=negativos.tipo)[["documento", "texto", "real"]],
], ignore_index=True)

veredictos = [validar(d, t) for d, t in zip(etiquetado.documento, etiquetado.texto)]
etiquetado["predicho"] = [v.tipo if v else "invalido" for v in veredictos]
etiquetado["confianza"] = [v.confianza if v else None for v in veredictos]
etiquetado["aceptado"] = [aceptar(v) for v in veredictos]
etiquetado["es_criterio"] = etiquetado.real == "criterio"

vp = (etiquetado.aceptado & etiquetado.es_criterio).sum()
fp = (etiquetado.aceptado & ~etiquetado.es_criterio).sum()
fn = (~etiquetado.aceptado & etiquetado.es_criterio).sum()
print(api)
print(f"Precisión: {vp / max(vp + fp, 1):.2f} · Exhaustividad: {vp / max(vp + fn, 1):.2f}  (VP={vp}, FP={fp}, FN={fn})")
pd.crosstab(etiquetado.real, etiquetado.predicho)

In [ ]:
# Los errores, con la razón del validador
errores = etiquetado[etiquetado.aceptado != etiquetado.es_criterio]
for (_, f), v in zip(errores.iterrows(), [veredictos[i] for i in errores.index]):
    print(f"[real: {f.real} · predicho: {f.predicho} · conf: {f.confianza}]\n  «{f.texto[:120]}»\n  razón: {v.razon if v else '-'}\n")

> **Para discutir:** mueve el `umbral` de `aceptar` entre 0.5 y 0.95 y vuelve a correr la medición (la caché hace que no cueste). ¿Qué pasa con la precisión y la exhaustividad? Para una base que consultan abogados, ¿qué error es más caro: un hecho que se cuela o un criterio que se pierde?

## 2 · El gate de fidelidad

Un criterio puede ser real y estar mal resumido. La falla típica es la **generalización no sostenida**: la cita dice «puede», la abstracción dice «siempre»; la cita habla de un mercado, la abstracción de todos.

El gate es otra llamada al modelo. En producción no conviene pagarla para cada criterio, así que primero filtramos los **criterios de riesgo**.

### ✏️ Ejercicio 2 · ¿Qué abstracciones son de riesgo?
Completa `es_de_riesgo`: devuelve `True` si la abstracción contiene lenguaje universal o absoluto (por ejemplo «todo», «toda», «todos», «siempre», «nunca», «cualquier», «ningún»). Usa una expresión regular con límites de palabra.

In [ ]:
RIESGO = re.compile(r"\b(todo|toda|todos|todas|siempre|nunca|cualquier|ning[uú]n|ninguna)\b", re.IGNORECASE)

def es_de_riesgo(abstraccion):
    return bool(RIESGO.search(abstraccion))

In [ ]:
class Fidelidad(BaseModel):
    etiqueta: Literal[("fiel", "generalizacion_no_sostenida")] = Field(description="Veredicto de fidelidad.")
    razon: str = Field(description="Una oración: qué afirma la abstracción que la cita no sostiene, o por qué es fiel.")

SISTEMA_FID = """Compara una ABSTRACCIÓN con la CITA y el APARTADO de donde proviene.
Es fiel si no afirma nada que el texto no sostenga: puede reformular y omitir detalles.
Es generalización no sostenida si amplía el alcance (de "puede" a "siempre", de un caso a todos),
convierte un indicador en una regla, o agrega consecuencias que el texto no establece."""

HERR_FID = {"name": "evaluar_fidelidad", "description": "Registra el veredicto de fidelidad.",
            "input_schema": esquema(Fidelidad)}

def gate(documento, cita, abstraccion):
    ctx = contexto(documento, cita)
    usuario = f"APARTADO:\n{ctx['texto'] if ctx else ''}\n\nCITA:\n{cita}\n\nABSTRACCIÓN:\n{abstraccion}"
    try:
        return Fidelidad.model_validate(llamar(SISTEMA_FID, usuario, HERR_FID))
    except ValidationError:
        return None

In [ ]:
fuente = gold.set_index("id_criterio")
filas = []
for _, p in pares.iterrows():
    g = fuente.loc[p.criterio_fuente]
    riesgo = es_de_riesgo(p.abstraccion)
    v = gate(g.documento, g.cita_literal, p.abstraccion) if riesgo else None
    # Sin riesgo no se revisa y pasa como fiel; si el gate falla, se descarta (fallar cerrado)
    final = "fiel" if not riesgo else (v.etiqueta if v else "descartado")
    filas.append({"id": p.id, "abstraccion": p.abstraccion, "real": p.etiqueta,
                  "de_riesgo": riesgo, "veredicto": final, "razon": v.razon if v else ""})

fid = pd.DataFrame(filas)
fid["correcto"] = fid.real == fid.veredicto
print(api, "\n")
print(f"Aciertos: {fid.correcto.sum()} de {len(fid)} · llamadas al gate: {fid.de_riesgo.sum()} de {len(fid)}")
fid[["id", "de_riesgo", "real", "veredicto", "correcto"]]

> **Para discutir:** busca el par que el filtro de riesgo dejó pasar sin revisar. No usa ninguna palabra universal y aun así generaliza: convierte un indicador en una regla. ¿Agregarías otra señal al filtro? ¿Cuánto cuesta revisar *todo* contra el riesgo de que algo así llegue a la base?

In [ ]:
# Contraprueba: ¿qué habría dicho el gate si revisáramos los 6 pares?
todos = [gate(fuente.loc[p.criterio_fuente].documento, fuente.loc[p.criterio_fuente].cita_literal, p.abstraccion)
         for _, p in pares.iterrows()]
fid["gate_sin_filtro"] = [v.etiqueta if v else "descartado" for v in todos]
print(api)
print("Aciertos revisando todo:", (fid.real == fid.gate_sin_filtro).sum(), "de", len(fid))
fid[["id", "real", "veredicto", "gate_sin_filtro"]]

## 3 · Aplicar ambos controles a tu extracción del Lab 2

Si corriste el Lab 2, pasamos sus criterios verificados por el validador y por el gate. El resultado es la base que usaremos en el Lab 3. Si no lo corriste, la celda usa el gold como respaldo.

In [ ]:
ruta_lab2 = SALIDAS / "criterios_lab2.jsonl"
if ruta_lab2.exists():
    base = pd.read_json(ruta_lab2, lines=True)
    base = base[base.estado == "verificado"].copy()
    origen = "Lab 2"
else:
    base = gold.copy()
    origen = "gold (respaldo)"

v_val = [validar(d, c) for d, c in zip(base.documento, base.cita_literal)]
base["tipo_validador"] = [v.tipo if v else "invalido" for v in v_val]
base["valido"] = [aceptar(v) for v in v_val]
base["de_riesgo"] = [es_de_riesgo(a) for a in base.criterio]
v_fid = [gate(d, c, a) if (ok and r) else None
         for d, c, a, ok, r in zip(base.documento, base.cita_literal, base.criterio, base.valido, base.de_riesgo)]
# no_aplica: no pasó el validador · sin_riesgo: no requirió gate · fiel / generalizacion_no_sostenida / descartado: veredicto del gate
base["fidelidad"] = ["no_aplica" if not ok else ("sin_riesgo" if not r else (v.etiqueta if v else "descartado"))
                     for ok, r, v in zip(base.valido, base.de_riesgo, v_fid)]
base["aceptado"] = base.valido & base.fidelidad.isin(["fiel", "sin_riesgo"])

base.to_json(SALIDAS / "criterios_validados.jsonl", orient="records", lines=True, force_ascii=False)
print(f"Origen: {origen} · {api}")
print(f"{len(base)} candidatos → {base.valido.sum()} pasan el validador → {base.aceptado.sum()} aceptados")
base[~base.aceptado][["id_chunk" if "id_chunk" in base else "documento", "tipo_validador", "fidelidad", "cita_literal"]]

## Cierre

| Lo que hicimos | Lo que en Norma+ es |
|---|---|
| Validador con siete categorías y contexto del apartado | Auditor y validador de proposiciones |
| `aceptar` con umbral | Fallar cerrado |
| Medir contra positivos y negativos etiquetados | Evaluación de cada componente, no solo del resultado final |
| `es_de_riesgo` + gate | Gate de fidelidad aplicado solo a criterios de riesgo |

**Para ciencias sociales:** el mismo patrón sirve siempre que un modelo codifique texto: primero clasifica, luego un segundo control con criterios explícitos, y ambos medidos contra una muestra codificada por personas. Es la versión automatizada de la confiabilidad entre codificadores.

**Siguiente:** en el Lab 3, la base tiene criterios repetidos entre documentos. Los encontraremos con embeddings y decidiremos con reglas cuáles fusionar.